In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
# from FEATURES.featuresV2 import *
# from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

In [4]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

In [ ]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

In [5]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]
dfs_data.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
244,Underdog,player_points,Jalen Brunson,Over,26.5,-137,2025-11-12,2025-11-12T00:17:54Z
245,Underdog,player_points,Jalen Brunson,Under,26.5,-137,2025-11-12,2025-11-12T00:17:54Z
246,Underdog,player_points,Karl-Anthony Towns,Over,21.5,-137,2025-11-12,2025-11-12T00:17:54Z
247,Underdog,player_points,Karl-Anthony Towns,Under,21.5,-137,2025-11-12,2025-11-12T00:17:54Z
248,Underdog,player_points,Ja Morant,Over,21.5,-137,2025-11-12,2025-11-12T00:17:54Z


In [8]:
from math import log, exp
from scipy.stats import poisson
import numpy as np
from nba_api.stats.endpoints import leaguedashteamstats

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()

def cap_factor(factor, min_val=0.85, max_val=1.15):
    """Cap adjustment factors to prevent extreme values"""
    return max(min_val, min(max_val, factor))

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    team_or = team_stats.at[player_team, 'OFF_RATING']
    team_pace = team_stats.at[player_team, 'PACE']
    opp_dr = team_stats.at[opp_team_id, 'DEF_RATING']
    opp_pace = team_stats.at[opp_team_id, 'PACE']

    df = s26
    try:
        player_df = df[df["PLAYER_NAME"] == PLAYER].copy()
        player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy()
        if len(player_df) < 5:  
            lambda_base = player_df_25['PTS'].mean()
            print(f"\n{PLAYER} - Baseline from Season 25: {lambda_base:.2f} pts ({len(player_df)} games)")
        else:
            lambda_base = player_df['PTS'].mean()
            print(f"\n{PLAYER} - Baseline from Season 26: {lambda_base:.2f} pts ({len(player_df)} games)")
    except:
        lambda_base = player_df['PTS'].mean()
        print(f"\n{PLAYER} - Baseline from Season 26: {lambda_base:.2f} pts ({len(player_df)} games)")

    if lambda_base <= 0:
        print(f"Skipping {PLAYER} - invalid baseline lambda")
        continue

    # Team offensive strength relative to league
    team_or_factor = cap_factor(team_or / league_avg_off_rtg)

    # Opponent defensive weakness relative to league (flip: lower def rating helps offense)
    opp_dr_factor = cap_factor(league_avg_def_rtg / opp_dr)
    
    # Pace adjustment relative to league
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = cap_factor(expected_pace / league_avg_pace)

    # Home court advantage (typical ~3% boost)
    home_factor = cap_factor(1.03 if home_flag else 0.97)

    # Recent form adjustment (last 5 games vs season average)
    recent_avg = player_df['PTS'].tail(5).mean()
    season_avg = player_df['PTS'].mean()
    form_factor = cap_factor(recent_avg / season_avg if season_avg > 0 else 1.0)

    # Minutes adjustment (last 5 vs season)
    recent_min_avg = player_df['MIN'].tail(5).mean()
    season_min_avg = player_df['MIN'].mean()
    min_factor = cap_factor(recent_min_avg / season_min_avg if season_min_avg > 0 else 1.0)

    # Usage rate adjustment (last 5 vs season)
    recent_usg_avg = player_df['USG_PCT'].tail(5).mean()
    season_usg_avg = player_df['USG_PCT'].mean()
    usg_factor = cap_factor(recent_usg_avg / season_usg_avg if season_usg_avg > 0 else 1.0)

    # Combine all factors
    combined_factor = (team_or_factor * 
                      opp_dr_factor * 
                      pace_factor * 
                      home_factor * 
                      form_factor * 
                      min_factor * 
                      usg_factor)

    # Adjust lambda
    lambda_adjusted = lambda_base * combined_factor

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'LINE_TYPE': line_type,
        'BASELINE_LAMBDA': lambda_base,
        'ADJUSTED_LAMBDA': lambda_adjusted,
        'COMBINED_FACTOR': combined_factor,
        'OVER%': prob_over_poisson,
        'UNDER%': 1 - prob_over_poisson,
        'IMPLIED_ODDS': 1 / prob_over_poisson if prob_over_poisson > 0 else None,
        'HOME': home_flag,
        'OPPONENT': opp_team
    })


Jalen Brunson - Baseline from Season 26: 27.22 pts (9 games)

Karl-Anthony Towns - Baseline from Season 26: 20.67 pts (9 games)

Ja Morant - Baseline from Season 26: 19.20 pts (10 games)

OG Anunoby - Baseline from Season 26: 18.22 pts (9 games)

Josh Hart - Baseline from Season 26: 8.25 pts (8 games)

Kentavious Caldwell-Pope - Baseline from Season 26: 9.18 pts (11 games)

Landry Shamet - Baseline from Season 26: 7.00 pts (9 games)

Cedric Coward - Baseline from Season 26: 14.82 pts (11 games)

Cam Spencer - Baseline from Season 26: 9.09 pts (11 games)

Immanuel Quickley - Baseline from Season 26: 14.40 pts (10 games)

Terance Mann - Baseline from Season 26: 9.80 pts (10 games)

Gradey Dick - Baseline from Season 26: 7.80 pts (10 games)

Jamal Shead - Baseline from Season 26: 6.40 pts (10 games)

Ziaire Williams - Baseline from Season 26: 10.00 pts (8 games)

Tyrese Martin - Baseline from Season 26: 7.10 pts (10 games)

Anfernee Simons - Baseline from Season 26: 15.00 pts (11 games)


In [9]:
prob_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
prob_df.head(10)

,NAME,LINE,LINE_TYPE,BASELINE_LAMBDA,ADJUSTED_LAMBDA,COMBINED_FACTOR,OVER%,UNDER%,IMPLIED_ODDS,HOME,OPPONENT
0,Russell Westbrook,12.5,X.5 (need 13+),15.000000,21.764420,1.450961,0.983023,0.016977,1.017270,1,DEN
1,Cade Cunningham,29.5,X.5 (need 30+),27.454545,37.961300,1.382696,0.919461,0.080539,1.087593,1,CHI
2,OG Anunoby,16.5,X.5 (need 17+),18.222222,22.913892,1.257470,0.915299,0.084701,1.092539,1,MEM
3,Trendon Watford,10.5,X.5 (need 11+),9.857143,15.756911,1.598527,0.913929,0.086071,1.094177,1,BOS
4,Kel'el Ware,10.5,X.5 (need 11+),10.090909,15.552536,1.541242,0.906015,0.093985,1.103735,1,CLE
5,Cam Spencer,8.5,X.5 (need 9+),9.090909,12.666509,1.393316,0.883985,0.116015,1.131241,0,NYK
6,Svi Mykhailiuk,8.5,X.5 (need 9+),9.300000,12.625523,1.357583,0.881844,0.118156,1.133987,1,IND
7,Isaac Okoro,7.5,X.5 (need 8+),7.900000,11.223827,1.420738,0.870684,0.129316,1.148523,0,DET
8,Amen Thompson,17.5,X.5 (need 18+),17.555556,22.761948,1.296567,0.867238,0.132762,1.153086,1,WAS
9,Tre Jones,13.5,X.5 (need 14+),13.500000,18.073431,1.338773,0.861104,0.138896,1.161300,0,DET
